# Graph RAG with AWS OpenSearch & Bedrock

## 📖 Overview

This notebook demonstrates **Graph-based Retrieval-Augmented Generation (Graph RAG)** using AWS services:
- **AWS OpenSearch Serverless** for storing graph nodes and edges
- **Amazon Bedrock Claude** for entity extraction and graph building
- **Amazon Bedrock Titan** for embeddings

### What is Graph RAG?

Graph RAG enhances traditional RAG by:
1. **Extract**: Identify entities and relationships from documents
2. **Build**: Construct a knowledge graph
3. **Store**: Index both nodes (entities) and edges (relationships)
4. **Traverse**: Navigate the graph to find connected context
5. **Generate**: Answer using graph-enriched context

### When to Use

✅ **Good for:**
- Multi-hop reasoning ("How is A related to C through B?")
- Relationship-focused queries
- Knowledge graphs
- Understanding connections and dependencies
- Domain-specific entity relationships

❌ **Not ideal for:**
- Simple factual lookup (overkill)
- Unstructured exploratory search
- When entity extraction is unreliable
- Very large graphs without pruning

### Architecture

```mermaid
graph TB
    subgraph "Indexing"
        A[Documents] --> B[Extract Entities<br/>Claude]
        B --> C[Build Knowledge Graph]
        C --> D[Store in OpenSearch<br/>Nodes + Edges]
    end

    subgraph "Querying"
        E[User Query] --> F[Identify Query Entities<br/>Claude]
        F --> G[Graph Traversal<br/>Find Connected Nodes]
        D -.-> G
        G --> H[Retrieve Context]
        H --> I[Generate Answer<br/>Claude]
    end

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#e1f5ff
    style F fill:#fff3e0
    style G fill:#f3e5f5
    style H fill:#e8f5e9
    style I fill:#ffe0b2
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Set, Tuple
import networkx as nx
import matplotlib.pyplot as plt

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
NODES_INDEX = 'graph_nodes'
EDGES_INDEX = 'graph_edges'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
LLM_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'

# Graph Parameters
MAX_HOPS = 2  # Maximum graph traversal depth
TOP_K_NODES = 10  # Initial nodes to retrieve

print(f"Configuration:")
print(f"  Region: {AWS_REGION}")
print(f"  Nodes Index: {NODES_INDEX}")
print(f"  Edges Index: {EDGES_INDEX}")
print(f"  Max Hops: {MAX_HOPS}")

## 3️⃣ Initialize Services

In [ ]:
# Initialize OpenSearch for nodes
nodes_manager = OpenSearchManager(
    region=AWS_REGION,
    index_name=NODES_INDEX
)

# Initialize OpenSearch for edges
edges_manager = OpenSearchManager(
    region=AWS_REGION,
    index_name=EDGES_INDEX
)

# Initialize Bedrock clients
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
llm = BedrockLLM(AWS_REGION, LLM_MODEL)

print("✓ Services initialized")

## 4️⃣ Entity Extraction

Extract entities and relationships from documents using Claude.

### Extraction Strategy

We use Claude to:
- Identify named entities (people, places, concepts)
- Extract relationships between entities
- Classify relationship types (e.g., "works_for", "located_in")

This structured extraction allows us to build a queryable graph.

In [ ]:
def extract_entities_and_relations(text: str) -> Dict:
    """
    Extract entities and relationships from text using Claude
    
    Returns:
        {
            'entities': [{'name': str, 'type': str, 'description': str}, ...],
            'relationships': [{'source': str, 'target': str, 'type': str}, ...]
        }
    """
    extraction_prompt = f"""
Extract entities and relationships from the following text.

Text:
{text}

Return your answer as a JSON object with this exact structure:
{{
    "entities": [
        {{"name": "Entity Name", "type": "PERSON|ORGANIZATION|LOCATION|CONCEPT", "description": "brief description"}}
    ],
    "relationships": [
        {{"source": "Entity1", "target": "Entity2", "type": "relationship_type"}}
    ]
}}

Only return the JSON, no other text.
"""
    
    try:
        response = llm.generate(extraction_prompt, temperature=0.1)
        # Extract JSON from response
        start_idx = response.find('{')
        end_idx = response.rfind('}') + 1
        json_str = response[start_idx:end_idx]
        return json.loads(json_str)
    except Exception as e:
        print(f"Extraction error: {e}")
        return {'entities': [], 'relationships': []}

# Test extraction
test_text = "AWS Bedrock provides access to foundation models. Amazon developed Bedrock to simplify AI adoption. The service is available in the us-west-2 region."

print("Testing entity extraction...\n")
result = extract_entities_and_relations(test_text)

print(f"Entities found: {len(result['entities'])}")
for entity in result['entities']:
    print(f"  - {entity['name']} ({entity['type']}): {entity['description']}")

print(f"\nRelationships found: {len(result['relationships'])}")
for rel in result['relationships']:
    print(f"  - {rel['source']} --[{rel['type']}]--> {rel['target']}")

## 5️⃣ Build Knowledge Graph

Process documents and construct the knowledge graph.

### Graph Structure

- **Nodes**: Entities with embeddings for semantic search
- **Edges**: Typed relationships between entities
- **Storage**: Both stored in separate OpenSearch indices

This dual-index approach allows:
- Fast entity lookup via vector search
- Efficient graph traversal via edge queries

In [ ]:
# Sample documents
sample_docs = [
    "AWS Bedrock is a managed service that provides access to foundation models from Anthropic, Amazon, and other AI companies through a single API.",
    "Anthropic developed Claude, a large language model family that includes Claude Opus, Sonnet, and Haiku variants.",
    "Amazon OpenSearch is a distributed search and analytics engine. OpenSearch Serverless is a serverless deployment option.",
    "RAG (Retrieval-Augmented Generation) combines information retrieval with text generation. It's commonly used with Claude and OpenSearch."
]

# Extract entities from all documents
all_entities = {}
all_relationships = []

print("Extracting entities and relationships...\n")
for i, doc in enumerate(sample_docs, 1):
    print(f"Processing document {i}/{len(sample_docs)}")
    result = extract_entities_and_relations(doc)
    
    # Collect unique entities
    for entity in result['entities']:
        if entity['name'] not in all_entities:
            all_entities[entity['name']] = entity
    
    # Collect relationships
    all_relationships.extend(result['relationships'])

print(f"\n✓ Extracted {len(all_entities)} unique entities")
print(f"✓ Extracted {len(all_relationships)} relationships")

## 6️⃣ Create Graph Indices

In [ ]:
# Create nodes index
nodes_manager.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    index_name=NODES_INDEX,
    force_recreate=True
)

# Create edges index (no vectors needed)
# For edges, we'll use a simpler structure without embeddings
edges_manager.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    index_name=EDGES_INDEX,
    force_recreate=True
)

print("✓ Graph indices created")

## 7️⃣ Index Graph Data

Generate embeddings and index nodes and edges.

In [ ]:
# Index nodes with embeddings
print("Indexing nodes...")
node_docs = []

for entity_name, entity_data in all_entities.items():
    # Create text representation for embedding
    entity_text = f"{entity_name}: {entity_data['description']} (Type: {entity_data['type']})"
    embedding = embedder.embed_text(entity_text)
    
    node_docs.append({
        'text': entity_text,
        'embedding': embedding,
        'metadata': {
            'entity_name': entity_name,
            'entity_type': entity_data['type'],
            'description': entity_data['description']
        }
    })

nodes_indexed = nodes_manager.index_documents(node_docs)
print(f"✓ Indexed {nodes_indexed} nodes")

# Index edges
print("\nIndexing edges...")
edge_docs = []

for rel in all_relationships:
    edge_text = f"{rel['source']} {rel['type']} {rel['target']}"
    embedding = embedder.embed_text(edge_text)
    
    edge_docs.append({
        'text': edge_text,
        'embedding': embedding,
        'metadata': {
            'source': rel['source'],
            'target': rel['target'],
            'relationship_type': rel['type']
        }
    })

edges_indexed = edges_manager.index_documents(edge_docs)
print(f"✓ Indexed {edges_indexed} edges")

## 8️⃣ Visualize Knowledge Graph

Visualize the constructed knowledge graph using NetworkX.

In [ ]:
# Build NetworkX graph
G = nx.DiGraph()

# Add nodes
for entity_name, entity_data in all_entities.items():
    G.add_node(entity_name, entity_type=entity_data['type'])

# Add edges
for rel in all_relationships:
    if rel['source'] in G and rel['target'] in G:
        G.add_edge(rel['source'], rel['target'], relationship=rel['type'])

# Visualize
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=2, iterations=50)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=2000, node_color='lightblue', alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')

# Draw edges
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, width=2)

# Draw edge labels
edge_labels = nx.get_edge_attributes(G, 'relationship')
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=6)

plt.title("Knowledge Graph", fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\nGraph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.3f}")

## 9️⃣ Graph Traversal RAG

Query the graph using multi-hop traversal.

### Query Process

1. **Identify**: Extract entities from query
2. **Search**: Find starting nodes via vector search
3. **Traverse**: Follow edges to connected nodes (up to MAX_HOPS)
4. **Context**: Gather all connected entities and relationships
5. **Generate**: Use graph context for answer generation

In [ ]:
def graph_rag_query(query: str, max_hops: int = 2) -> Tuple[str, List[str]]:
    """
    Query using graph traversal
    
    Returns:
        (answer, context_entities)
    """
    print(f"Query: {query}\n")
    
    # Step 1: Find starting entities
    query_embedding = embedder.embed_text(query)
    starting_nodes = nodes_manager.vector_search(query_embedding, top_k=3)
    
    print(f"Starting entities:")
    entity_names = set()
    for node in starting_nodes:
        entity_name = node['metadata']['entity_name']
        entity_names.add(entity_name)
        print(f"  - {entity_name} (score: {node['score']:.3f})")
    
    # Step 2: Traverse graph
    print(f"\nTraversing graph (max {max_hops} hops)...")
    visited = entity_names.copy()
    context_parts = []
    
    for hop in range(max_hops):
        # Find edges from current entities
        new_entities = set()
        
        for entity in list(visited):
            # Search for edges where this entity is source or target
            edge_results = edges_manager.client.search(
                index=EDGES_INDEX,
                body={
                    "query": {
                        "bool": {
                            "should": [
                                {"match": {"metadata.source": entity}},
                                {"match": {"metadata.target": entity}}
                            ]
                        }
                    },
                    "size": 10
                }
            )
            
            for hit in edge_results['hits']['hits']:
                meta = hit['_source']['metadata']
                context_parts.append(f"{meta['source']} {meta['relationship_type']} {meta['target']}")
                new_entities.add(meta['source'])
                new_entities.add(meta['target'])
        
        visited.update(new_entities)
        print(f"  Hop {hop+1}: Found {len(new_entities)} new entities")
    
    print(f"\nTotal context: {len(context_parts)} relationships")
    
    # Step 3: Generate answer
    context = "\n".join(context_parts)
    
    prompt = f"""
Based on the following knowledge graph relationships, answer the question.

Knowledge Graph:
{context}

Question: {query}

Answer:
"""
    
    answer = llm.generate(prompt)
    
    return answer, list(visited)

# Test queries
test_queries = [
    "How is AWS Bedrock related to Claude?",
    "What services are connected to OpenSearch?",
    "Tell me about the relationship between Anthropic and AWS"
]

for i, question in enumerate(test_queries, 1):
    print(f"\n{'='*70}")
    print(f"Test Query {i}")
    print('='*70)
    
    answer, entities = graph_rag_query(question, max_hops=MAX_HOPS)
    
    print(f"\n📝 Answer:")
    print(answer)
    print(f"\n🔗 Entities used: {', '.join(list(entities)[:5])}...")

## 🔟 Compare with Simple RAG

Let's compare Graph RAG vs Simple RAG for the same query.

In [ ]:
comparison_query = "What is the connection between Anthropic and AWS Bedrock?"

print("GRAPH RAG:")
print("="*70)
graph_answer, graph_entities = graph_rag_query(comparison_query, max_hops=2)
print(f"\nAnswer: {graph_answer}")
print(f"Context: {len(graph_entities)} entities")

print("\n\nSIMPLE RAG (for comparison):")
print("="*70)
# Simple vector search on original documents
query_emb = embedder.embed_text(comparison_query)
simple_results = nodes_manager.vector_search(query_emb, top_k=3)
simple_context = "\n".join([r['text'] for r in simple_results])
simple_answer = llm.generate_with_context(comparison_query, [r['text'] for r in simple_results])
print(f"Answer: {simple_answer}")
print(f"Context: {len(simple_results)} chunks")

print("\n\nKEY DIFFERENCES:")
print("- Graph RAG: Explores relationships and connected entities")
print("- Simple RAG: Only retrieves similar text chunks")
print("- Graph RAG is better for: Multi-hop reasoning, relationship queries")
print("- Simple RAG is better for: Direct factual lookup, speed")

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ Entity extraction with Claude
✅ Knowledge graph construction
✅ Dual-index storage (nodes + edges)
✅ Multi-hop graph traversal
✅ Graph-enhanced answer generation

### Performance Characteristics

- **Latency**: 3-5 seconds per query (entity extraction + traversal)
- **Cost**: Higher due to extraction step (~$0.10-0.20 per query)
- **Accuracy**: Better for relationship queries
- **Complexity**: Higher indexing overhead

### When to Use Graph RAG

**Use Graph RAG when:**
- Queries involve relationships ("How is X related to Y?")
- Multi-hop reasoning is required
- Domain has clear entity structure
- Understanding connections is important

**Use Simple RAG when:**
- Queries are direct factual lookup
- Speed is critical
- Entity extraction is unreliable
- Documents lack clear structure

### Limitations

1. **Extraction quality**: Depends on LLM's entity recognition
2. **Graph size**: Can become expensive for large graphs
3. **Traversal cost**: More hops = more API calls
4. **Maintenance**: Graph needs updating as documents change

### Next Steps

- **Hybrid approach**: Combine graph + vector search
- **Entity linking**: Resolve entity coreferences
- **Graph pruning**: Limit graph size with importance scoring
- **Incremental updates**: Update graph without full rebuild

## 🧹 Cleanup

In [ ]:
# Uncomment to delete indices
# nodes_manager.delete_index(NODES_INDEX)
# edges_manager.delete_index(EDGES_INDEX)
# print("✓ Deleted graph indices")

print("\nTo delete indices later:")
print(f"  nodes_manager.delete_index('{NODES_INDEX}')")
print(f"  edges_manager.delete_index('{EDGES_INDEX}')")